In [9]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler


In [10]:
df = pd.read_csv("joined_data.csv")
df.head()

,observation_date,gdp_growth_rate,unemployment_rate,cpi_commodities,covid_dummy
0,1956-01-01,-1.5,4.033333,31.166667,0
1,1956-04-01,3.3,4.200000,31.400000,0
2,1956-07-01,-0.3,4.133333,31.766667,0
3,1956-10-01,6.8,4.133333,32.033333,0
4,1957-01-01,2.6,3.933333,32.233333,0


In [11]:
# getting rid of covid quarters
df = df[~df['observation_date'].isin(['2020-04-01', '2020-07-01'])]

In [12]:
# specifying X and Y for later models 
X = df[['gdp_growth_rate', 'cpi_commodities', 'covid_dummy']]
y = df['unemployment_rate']

In [13]:
# baseline model with just simple mean
baseline_prediction = y.mean()

# 2. Create an array of predictions (all values equal to the mean)
y_pred_baseline = np.full(shape=y.shape, fill_value=baseline_prediction)

rmse_baseline = (root_mean_squared_error(y, y_pred_baseline))

print(f"Baseline Prediction (Mean): {baseline_prediction:.4f}")
print(f"Baseline RMSE: {rmse_baseline:.4f}")

Baseline Prediction (Mean): 5.7876
Baseline RMSE: 1.6073


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, shuffle=False
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred = lr.predict(X_test_scaled)

rmse = (root_mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"R^2: {r2:.4f}")

print("model coefs")
for name, coef in zip(X.columns, lr.coef_):
    print(f"  {name}: {coef:.4f}")
print(f"  Intercept: {lr.intercept_:.4f}")

RMSE: 2.6570
R²: -10.8519
model coefs
  gdp_growth_rate: -0.0402
  cpi_commodities: 0.4344
  covid_dummy: 0.0000
  Intercept: 6.0438


In [ ]:
df_diff = df.copy()

df_diff['cpi_diff'] = df_diff['cpi_commodities'].diff()
df_diff['unemployment_diff'] = df_diff['unemployment_rate'].diff()

df_diff = df_diff.dropna()

X = df_diff[['gdp_growth_rate', 'cpi_diff']]
y = df_diff['unemployment_diff']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, shuffle=False
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred = lr.predict(X_test_scaled)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)


print(f"RMSE: {rmse:.4f}")
print(f"R^2: {r2:.4f}")

print("model coefs")
for name, coef in zip(X.columns, lr.coef_):
    print(f"  {name}: {coef:.4f}")
print(f"  Intercept: {lr.intercept_:.4f}")

RMSE: 0.5317
R²: -0.0540
model coefs
  gdp_growth_rate: -0.2394
  cpi_diff: -0.0114
  Intercept: 0.0071


In [17]:
# Creating model with lagged features
df_diff['gdp_lag1'] = df_diff['gdp_growth_rate'].shift(1)
df_diff['cpi_diff_lag1'] = df_diff['cpi_diff'].shift(1)
df_diff['unemployment_diff_lag1'] = df_diff['unemployment_diff'].shift(1)

df_final = df_diff.dropna().copy()

X = df_final[['gdp_lag1', 'cpi_diff_lag1', 'unemployment_diff_lag1']]
y = df_final['unemployment_diff']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, shuffle=False
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred = lr.predict(X_test_scaled)

print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.4f}")
print(f"R^2: {r2_score(y_test, y_pred):.4f}")

Lagged Model RMSE: 0.5282
Lagged Model R²: -0.0401


In [19]:
df_diff

,observation_date,gdp_growth_rate,unemployment_rate,cpi_commodities,covid_dummy,cpi_diff,unemployment_diff,gdp_lag1,cpi_diff_lag1,unemployment_diff_lag1
1,1956-04-01,3.3,4.200000,31.400000,0,0.233333,1.666667e-01,NaN,NaN,NaN
2,1956-07-01,-0.3,4.133333,31.766667,0,0.366667,-6.666667e-02,3.3,0.233333,1.666667e-01
3,1956-10-01,6.8,4.133333,32.033333,0,0.266667,0.000000e+00,-0.3,0.366667,-6.666667e-02
4,1957-01-01,2.6,3.933333,32.233333,0,0.200000,-2.000000e-01,6.8,0.266667,0.000000e+00
5,1957-04-01,-0.9,4.100000,32.433333,0,0.200000,1.666667e-01,2.6,0.200000,-2.000000e-01
...,...,...,...,...,...,...,...,...,...,...
275,2024-10-01,1.9,4.133333,222.827333,0,0.611000,-3.333333e-02,3.3,-1.212000,2.000000e-01
276,2025-01-01,-0.6,4.133333,224.413000,0,1.585667,8.881784e-16,1.9,0.611000,-3.333333e-02
277,2025-04-01,3.8,4.200000,223.847000,0,-0.566000,6.666667e-02,-0.6,1.585667,8.881784e-16
278,2025-07-01,4.4,4.333333,225.254000,0,1.407000,1.333333e-01,3.8,-0.566000,6.666667e-02
